# Geomagnetic Storm Prediction from Cosmic Ray Measurements

## Abstract
This project investigates whether ground-based cosmic ray neutron monitor data, combined with near-Earth solar wind parameters, can be used to forecast the intensity of geomagnetic storms several hours in advance. The target variable is the Disturbance Storm Time (Dst) index, a standard measure of geomagnetic activity. The domain is Space Weather — specifically CME-driven geomagnetic disturbances — over the period 1981–2023.  

Two data sources are used: hourly neutron count rates from the Lomnický štít station (LMKS), provided by Kisvárdai et al. (2024), and hourly solar wind parameters from the NASA OMNI dataset (Papitashvili & King, 2020). After joint cleaning and feature engineering, the merged dataset spans 364,728 hourly observations across solar cycles 21–25.

## Introduction

This project belongs to the field of *Space Weather* — the study of how solar activity affects near-Earth space and, consequently, technological infrastructure on the ground and in orbit. Severe geomagnetic storms can induce geomagnetically induced currents (GICs) in power grid infrastructure, disrupt satellite navigation and radio communications, accelerate orbital decay of low-Earth-orbit satellites, and pose radiation hazards to astronauts and high-altitude aviation. A forecast with several hours of lead time allows grid operators, satellite controllers, and mission planners to take protective action. The practical operational minimum is one hour; longer horizons of $6$–$24$ hours are actively sought by the space weather community `[ISH17]`.


### Domain Definition

A geomagnetic storm is not a single event but a chain of causally connected phenomena, each of which leaves a measurable trace.

**Stage 1 — Eruption at the Sun.** During periods of high solar activity, magnetically stressed regions on the solar surface can release large quantities of magnetized plasma into interplanetary space — a *coronal mass ejection* (CME). A CME is accompanied by a fast-forward shock wave propagating ahead of it. The overall level of solar activity follows an approximately $11$-year cycle, which means that the frequency and intensity of CMEs — and consequently of geomagnetic storms — is not constant over time but rises and falls with the cycle. 

<img src="images/fig1_chain.png" style="width:80%; max-width:700px"/>  

*Figure 1 — The Sun–L1–Earth chain. Synthetic illustration.*


**Stage 2 — Propagation through interplanetary space.** The CME and its preceding shock travel outward from the Sun at speeds ranging from $300$ to $2{,}000\ \text{km/s}$. As the shock propagates, it sweeps aside galactic cosmic rays — high-energy charged particles that arrive continuously at Earth from outside the solar system. This sweeping creates a temporary deficit of cosmic rays, called a **Forbush Decrease**, which can be detected at ground level by a neutron monitor hours before the CME itself arrives at Earth `[KIS25]`. The Forbush Decrease is therefore a precursor signal embedded in the cosmic ray record.  

<img src="images/fig2_forbush_dst.png" style="width:70%; max-width:600px"/>  

*Figure 2 — Synthetic Forbush Decrease and $D_{st}$ response. For illustration only.*  

**Stage 3 — Arrival at L1 and the solar wind parameters.** Before reaching Earth, the CME passes through the **L1 Lagrange point** — a gravitationally stable location in space approximately $1.5 \times 10^6\ \text{km}$ upstream of Earth, where the gravitational pull of the Sun and Earth balance the centrifugal force on an orbiting body. Spacecraft stationed at L1 have an unobstructed view of the incoming solar wind and measure its properties continuously in near-real time: the speed $V_{sw}\ [\text{km/s}]$, the proton density $n_p\ [\text{cm}^{-3}]$, the plasma temperature $T\ [\text{K}]$, and — most critically for storm prediction — the north-south component of the interplanetary magnetic field $B_z\ [\text{nT}]$ `[PAP20]`, `[KIN05]`. When $B_z$ is directed southward ($B_z < 0$) and remains so for an extended period, the solar wind couples efficiently to Earth's magnetosphere through a process called magnetic reconnection.

**Stage 4 — Energy injection into the magnetosphere.** The rate at which energy enters the magnetosphere depends on the interplanetary electric field:

$$E_y = -V_{sw} \cdot B_z \cdot 10^{-3} \quad [\text{mV/m}]$$

A large positive $E_y$ — produced by high solar wind speed combined with strong southward $B_z$ — drives energetic particles into a toroidal current circulating around Earth at altitudes of $2$–$9\ R_E$, known as the *ring current*. The intensification of this current depresses Earth's horizontal magnetic field at low latitudes. The classical description of this process is given by the Burton et al. (1975) equation `[BUR75]`:

$$\frac{dD_{st}}{dt} = Q(E_y) - \frac{D_{st}}{\tau}$$

where $Q(E_y)$ is the injection rate and $D_{st}/\tau$ is the natural decay of the ring current with characteristic time $\tau \approx 7$–$8$ hours. The injection function is:

$$Q(E_y) = \begin{cases} -4.4\,(E_y - 0.5) & \text{if } E_y > 0.5\ \text{mV/m} \\ 0 & \text{otherwise} \end{cases}$$

<img src="images/fig3_burton.png" style="width:60%; max-width:700px"/>  

*Figure 3 — Burton (1975) physical model `[BUR75]`. Synthetic $E_y(t)$ profile.*  


**Stage 5 — Ground-level response.** The depression of the horizontal magnetic field is recorded at a network of low-latitude observatories and averaged into the **Disturbance Storm Time ($D_{st}$) index** `[WDC]`, expressed in nanotesla ($\text{nT}$). This is the target variable of the present project. Storm severity is conventionally classified as:

$$D_{st} > -50\ \text{nT} \quad \text{(quiet)}$$
$$-100\ \text{nT} < D_{st} \leq -50\ \text{nT} \quad \text{(moderate storm)}$$
$$-200\ \text{nT} < D_{st} \leq -100\ \text{nT} \quad \text{(strong storm)}$$
$$D_{st} \leq -200\ \text{nT} \quad \text{(extreme storm)}$$

The decay time $\tau \approx 7$–$8$ hours from the Burton equation defines the natural timescale of storm evolution and motivates the choice of prediction horizons examined in this project. The physical travel time of a CME shock from the Sun to L1 ($1$–$4$ days) and the L1-to-Earth propagation time ($15$–$60$ minutes) jointly constrain the range of horizons for which ground-based and in-situ measurements can plausibly carry predictive information: between $1$ and approximately $72$ hours `[PAR10]`, `[ISH17]`.

### 2.2 Prior Work

The Burton et al. (1975) equation `[BUR75]` established the classical physical relationship between $E_y$ and $D_{st}$, providing a physically interpretable baseline that remains in use today. More recent machine learning approaches have extended this to multi-hour prediction: Parnowski (2010) `[PAR10]` reports $R^2 > 0.90$ at $1$–$6$ hour horizons using OMNI solar wind parameters alone. The MagNet competition `[NAI23]` benchmarked a broad range of models — from linear regression to deep neural networks — achieving RMSE in the range of $11$–$15\ \text{nT}$ at a $1$-hour horizon. Hu et al. (2023) `[HU23]` applied multi-fidelity boosted neural networks with explicit prediction intervals, further demonstrating the value of uncertainty quantification in operational forecasting.

**Gap.** Most published models rely exclusively on in-situ solar wind measurements at L1, which provide at most $15$–$60$ minutes of natural lead time before the disturbance reaches Earth. The Forbush Decrease — described in Stage 2 above — offers a physically distinct signal that precedes the storm's main phase by several hours. Kisvárdai et al. (2025) `[KIS25]` analyzed $42$ years of neutron monitor data from the Lomnický štít station and found a Pearson correlation of $r = 0.314$ between the neutron flux and $D_{st}$ at a time shift of $7$–$21$ hours, and a predictive power score of $\text{PPS} = 0.22$. This suggests that the cosmic ray signal carries information about future storm intensity beyond what the instantaneous solar wind parameters alone capture.

**Contribution.** This project investigates whether combining cosmic ray neutron flux measurements `[KIS24]` with OMNI solar wind parameters `[PAP20]` can extend the useful prediction horizon and improve forecast skill relative to OMNI-only models.

**Scope.** The analysis is restricted to CME-driven storms over the period $1981$–$2023$. The dataset is not filtered by storm driver type — Co-rotating Interaction Region (CIR) driven storms, which follow a different physical mechanism and do not produce a Forbush Decrease precursor, remain in the record and represent a source of noise for the cosmic ray features. This is acknowledged as a limitation of the present study.
